In [2]:
import pandas as pd
import numpy as np
import zipfile

# قراءة ملف بيانات الشقق مباشرة من داخل الملف المضغوط
with zipfile.ZipFile('riyadh_data.zip') as z:
    with z.open('riyadh_apartments_data.csv') as f:
        df = pd.read_csv(f)

# استعراض الأعمدة ونوع البيانات والقيم المفقودة
print(df.info())

# عرض أول 5 صفوف
df.head()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   City                  10000 non-null  str    
 1   Region                10000 non-null  str    
 2   Neighborhood          10000 non-null  str    
 3   Property Type         10000 non-null  str    
 4   Area (sqm)            10000 non-null  int64  
 5   Bedrooms              10000 non-null  int64  
 6   Bathrooms             10000 non-null  int64  
 7   Floor Number          10000 non-null  int64  
 8   Elevator              10000 non-null  str    
 9   Property Age (years)  10000 non-null  int64  
 10  Furnished             10000 non-null  str    
 11  Listing Type          10000 non-null  str    
 12  Selling Price (SAR)   10000 non-null  float64
dtypes: float64(1), int64(5), str(7)
memory usage: 1.4 MB
None


,City,Region,Neighborhood,Property Type,Area (sqm),Bedrooms,Bathrooms,Floor Number,Elevator,Property Age (years),Furnished,Listing Type,Selling Price (SAR)
0,Riyadh,West,Shubra,apartment,138,4,4,4,Yes,8,Yes,Sale,459219.35
1,Riyadh,West,Al-Oraija,apartment,274,4,2,2,Yes,8,Yes,Sale,810905.38
2,Riyadh,East,Al-Sulay,apartment,185,1,2,3,No,9,Yes,Sale,797967.69
3,Riyadh,North,Al-Yasmeen,apartment,272,4,2,3,No,6,No,Sale,2245339.91
4,Riyadh,West,Al-Oraija Al-Gharbi (Western Oraija),apartment,244,2,3,5,Yes,2,No,Sale,827890.53


In [3]:
# 1. حساب سعر المتر المربع
df['Price_per_sqm'] = df['Selling Price (SAR)'] / df['Area (sqm)']

# 2. ملخص إحصائي للأعمدة الرقمية (الأسعار، المساحة، المواصفات)
print("--- الملخص الإحصائي ---")
display(df.describe().round(2))

# 3. مقارنة متوسط الأسعار وسعر المتر حسب المنطقة
region_stats = df.groupby('Region').agg(
    Avg_Price=('Selling Price (SAR)', 'mean'),
    Avg_Price_per_sqm=('Price_per_sqm', 'mean'),
    Total_Listings=('Selling Price (SAR)', 'count')
).reset_index().sort_values(by='Avg_Price_per_sqm', ascending=False)

print("\n--- مقارنة مناطق الرياض ---")
display(region_stats.round(2))

--- الملخص الإحصائي ---


,Area (sqm),Bedrooms,Bathrooms,Floor Number,Property Age (years),Selling Price (SAR),Price_per_sqm
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,185.28,2.51,2.50,3.01,5.47,714450.32,3844.43
std,56.72,1.12,1.11,1.41,2.87,643373.17,3098.81
min,89.00,1.00,1.00,1.00,1.00,72401.86,800.37
25%,136.00,2.00,2.00,2.00,3.00,261800.96,1412.38
50%,184.00,3.00,2.00,3.00,5.00,469296.78,3027.07
75%,234.00,4.00,3.00,4.00,8.00,924126.26,4534.27
max,284.00,4.00,4.00,5.00,10.00,3384003.90,11997.42



--- مقارنة مناطق الرياض ---


,Region,Avg_Price,Avg_Price_per_sqm,Total_Listings
2,North,1767499.91,9450.89,2020
1,East,732998.37,3995.51,2047
4,West,555051.47,2984.67,1997
0,Central,277569.25,1500.74,1972
3,South,212780.75,1148.06,1964


In [4]:
# 1. تصنيف الشقق بناءً على سعر المتر المربع
def categorize_price(sqm_price):
    if sqm_price >= 8000:
        return 'فاخرة (Luxury)'
    elif sqm_price >= 3000:
        return 'متوسطة (Mid-tier)'
    else:
        return 'اقتصادية (Affordable)'

df['Price_Category'] = df['Price_per_sqm'].apply(categorize_price)

# 2. حفظ البيانات المجهزة في ملف CSV جديد جاهز لـ Power BI
output_file = 'riyadh_apartments_cleaned.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"تم تصدير الملف بنجاح باسم: {output_file}")
print("\nتوزيع فئات الأسعار:")
print(df['Price_Category'].value_counts())

تم تصدير الملف بنجاح باسم: riyadh_apartments_cleaned.csv

توزيع فئات الأسعار:
Price_Category
اقتصادية (Affordable)    4949
متوسطة (Mid-tier)        3453
فاخرة (Luxury)           1598
Name: count, dtype: int64
